# Install the libraries

In [2]:
!pip install -q unsloth
!pip install -q gTTS librosa soundfile

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.5.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


# Imports and a memory helper

In [ ]:
from unsloth import FastLanguageModel
import torch, gc, librosa
from transformers import pipeline

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def vram(tag=""):
    used = torch.cuda.memory_allocated() / 1024**3
    print(f"[{tag}] GPU memory in use: {used:.2f} GB")

print(torch.cuda.get_device_name(0))
vram("start")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1122 21:15:03.979000 1263 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 21:15:04.018000 1263 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Tesla T4
[start] GPU memory in use: 0.01 GB


In [ ]:
from gtts import gTTS

question = ("If a train travels at 60 kilometers per hour for two and a half hours, "
            "how far does it go? Explain your reasoning step by step.")
gTTS(question, lang="en").save("sample_query.mp3")

from IPython.display import Audio
Audio("sample_query.mp3")

# Transcribe with Whisper

In [ ]:
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",     # use "whisper-base" if memory is tight
    torch_dtype=torch.float16,
    device=0,
)
vram("whisper loaded")

# Encoding: load audio as 16 kHz mono, the format Whisper expects
def load_audio(path):
    wav, _ = librosa.load(path, sr=16000, mono=True)
    return {"raw": wav, "sampling_rate": 16000}

def transcribe(paths):
    audios = [load_audio(p) for p in paths]
    results = asr(audios, batch_size=8, chunk_length_s=30)  # batching + long-audio chunking
    return [r["text"].strip() for r in results]

texts = transcribe(["sample_query.mp3"])
print(texts[0])

# Free Whisper before loading the LLM
del asr
free_gpu()
vram("whisper freed")

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

... (7 outputs left out)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[whisper loaded] GPU memory in use: 0.46 GB


Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
Pa

If a train travels at 60 kilometers per hour for two and a half hours, how far does it go? Explain your reasoning step by step.
[whisper freed] GPU memory in use: 0.01 GB


In [ ]:
!ffmpeg -y -i q2.mp4 -ar 16000 -ac 1 q2_user.wav

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

My own recording

In [ ]:
from IPython.display import Audio
Audio("q2_user.wav")

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display


files = ["sample_query.mp3", "q2_user.wav"]

# Make sure Whisper is loaded
try:
    asr
except NameError:
    asr = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        torch_dtype=torch.float16,
        device=0,
    )
    vram("whisper reloaded")

# Transcribe BOTH files in one batch
files = ["sample_query.mp3", "q2_user.wav"]
texts = transcribe(files)

for f, t in zip(files, texts):
    print(f"{f}  ->  {t}\n")

# Free Whisper so the LLM fits in memory
del asr
free_gpu()
vram("whisper freed")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


[whisper reloaded] GPU memory in use: 0.46 GB
sample_query.mp3  ->  If a train travels at 60 kilometers per hour for two and a half hours, how far does it go? Explain your reasoning step by step.

q2_user.wav  ->  What is the capital of Bangladesh and why is it important?

[whisper freed] GPU memory in use: 0.01 GB


# Load the quantized reasoning model

In [ ]:
for name in ("model", "tokenizer"):
    if name in globals():
        del globals()[name]
free_gpu()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
vram("base + new LoRA")

==((====))==  Unsloth 2026.11.9: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
[base + new LoRA] GPU memory in use: 2.29 GB


# Pass the transcription to the LLM

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = "You are a careful assistant. Think step by step, then give a clear final answer."

def answer(questions, max_new_tokens=400):
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM},
             {"role": "user", "content": q}],
            tokenize=False, add_generation_prompt=True)
        for q in questions
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                       add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, use_cache=True)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)


replies = answer(texts)

for i, (q, a) in enumerate(zip(texts, replies), 1):
    print(f"--- Query {i} ---")
    print("TRANSCRIPT:", q)
    print("ANSWER:", a)
    print()

vram("after generation")

before_replies = replies    # saved for the before/after comparison

Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Query 1 ---
TRANSCRIPT: If a train travels at 60 kilometers per hour for two and a half hours, how far does it go? Explain your reasoning step by step.
ANSWER: To find the distance traveled by the train, we need to follow these steps:

1. **Identify the given information:**
   - Speed of the train: 60 kilometers per hour (km/h)
   - Time traveled: 2.5 hours

2. **Understand the formula to calculate distance:**
   The formula to calculate distance is: Distance = Speed × Time

3. **Apply the formula:**
   - Distance = Speed × Time
   - Distance = 60 km/h × 2.5 hours

4. **Calculate the distance:**
   - Distance = 60 × 2.5
   - Distance = 150 kilometers

Therefore, the train travels 150 kilometers in 2.5 hours.

--- Query 2 ---
TRANSCRIPT: What is the capital of Bangladesh and why is it important?
ANSWER: The capital of Bangladesh is Dhaka. 

Dhaka is the most populous city in Bangladesh, with over 20 million people residing within its metropolitan area. It is an important city for se

Fine-tuning dataset

In [ ]:
import re
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main", split="train[:1000]")

def fmt(ex):
    reasoning, final = ex["answer"].split("####")
    reasoning = re.sub(r"<<.*?>>", "", reasoning).strip()      # remove calculator tags
    reply = f"{reasoning}\n\nFinal answer: {final.strip()}"
    msgs = [{"role": "user", "content": ex["question"]},
            {"role": "assistant", "content": reply}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = ds.map(fmt)
print(ds[0]["text"][:600])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 22 Nov 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.

Final answer: 72<|eot_id|>


Train the adapter

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,                 # about 10-15 min on a T4; use 60 if GPU time is tight
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

torch.cuda.reset_peak_memory_stats()
stats = trainer.train()
print(f"Training time: {stats.metrics['train_runtime']/60:.1f} min")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.971739
20,1.164100
30,1.011540
40,0.951830
50,0.980213
60,0.920165
70,0.893444
80,0.913597
90,0.932235
100,0.971961


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.


Training time: 3.3 min
Peak GPU memory: 7.98 GB


In [ ]:
model.save_pretrained("task4_lora_adapter")
tokenizer.save_pretrained("task4_lora_adapter")

# Reload from disk to prove it works
del model, trainer
free_gpu()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="task4_lora_adapter",
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
vram("reloaded adapter")

Unsloth: Restored added_tokens_decoder metadata in task4_lora_adapter/tokenizer_config.json.


==((====))==  Unsloth 2026.11.9: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load task4_lora_adapter as a legacy tokenizer.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
[reloaded adapter] GPU memory in use: 3.09 GB


In [ ]:
!zip -r task4_lora_adapter.zip task4_lora_adapter
from google.colab import files
files.download("task4_lora_adapter.zip")

updating: task4_lora_adapter/ (stored 0%)
updating: task4_lora_adapter/chat_template.jinja (deflated 71%)
updating: task4_lora_adapter/tokenizer_config.json (deflated 96%)
updating: task4_lora_adapter/README.md (deflated 65%)
updating: task4_lora_adapter/adapter_model.safetensors (deflated 8%)
updating: task4_lora_adapter/adapter_config.json (deflated 60%)
updating: task4_lora_adapter/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
after_replies = answer(texts)

for q, b, a in zip(texts, before_replies, after_replies):
    print("QUESTION:", q)
    print("\nBEFORE TRAINING:\n", b)
    print("\nAFTER TRAINING:\n", a)
    print("\n" + "="*60 + "\n")

Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: If a train travels at 60 kilometers per hour for two and a half hours, how far does it go? Explain your reasoning step by step.

BEFORE TRAINING:
 To find the distance traveled by the train, we need to follow these steps:

1. **Identify the given information:**
   - Speed of the train: 60 kilometers per hour (km/h)
   - Time traveled: 2.5 hours

2. **Understand the formula to calculate distance:**
   The formula to calculate distance is: Distance = Speed × Time

3. **Apply the formula:**
   - Distance = Speed × Time
   - Distance = 60 km/h × 2.5 hours

4. **Calculate the distance:**
   - Distance = 60 × 2.5
   - Distance = 150 kilometers

Therefore, the train travels 150 kilometers in 2.5 hours.

AFTER TRAINING:
 To find the distance traveled by the train, we need to multiply the speed of the train by the time it travels.
Speed of the train = 60 kilometers per hour
Time = 2.5 hours
Distance = Speed x Time
= 60 x 2.5
= 150 kilometers

Final answer: 150


QUESTION: What is the 

# End-to-end demo

In [ ]:
import time

def speech_to_answer(audio_paths):
    global asr
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    # Stage 1: speech -> text
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
                   dtype=torch.float16, device=0)
    transcripts = transcribe(audio_paths)
    del asr
    free_gpu()

    # Stage 2: text -> reasoning
    replies = answer(transcripts)

    for p, t, a in zip(audio_paths, transcripts, replies):
        print(f"AUDIO: {p}\nHEARD: {t}\nANSWER: {a}\n{'-'*60}")
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB | "
          f"Time: {time.time()-t0:.1f}s")

speech_to_answer(["sample_query.mp3", "q2_user.wav"])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AUDIO: sample_query.mp3
HEARD: If a train travels at 60 kilometers per hour for two and a half hours, how far does it go? Explain your reasoning step by step.
ANSWER: To find the distance traveled by the train, we need to multiply the speed of the train by the time it travels.
Speed of the train = 60 kilometers per hour
Time = 2.5 hours
Distance = Speed x Time
= 60 x 2.5
= 150 kilometers

Final answer: 150
------------------------------------------------------------
AUDIO: q2_user.wav
HEARD: What is the capital of Bangladesh and why is it important?
ANSWER: The capital of Bangladesh is Dhaka. Dhaka is important because it is the country's largest city and serves as the country's economic, cultural, and administrative center. It is also the seat of the country's government and is home to many important institutions, including the parliament, the supreme court, and the president's office.

Dhaka is also a major hub for trade, commerce, and industry, and is home to many major industries, 

**Pipeline**: Whisper-small (ASR, fp16) → Llama-3.2-3B-Instruct (Unsloth dynamic 4-bit) with a LoRA adapter I fine-tuned → answer.

**Fine-tuning:** QLoRA (r=16) on 1,000 GSM8K rows, 100 steps, 3.3 min on a T4, peak VRAM 7.98 GB. Adapter saved to task4_lora_adapter, reloaded from disk and used for the demo.

**Encoding**: audio loaded as 16 kHz mono (the mp4 was converted with ffmpeg).

**Batching**: Whisper batch_size=8 and chunk_length_s=30; batched LLM generation with left padding.

**Memory**: Whisper freed before the LLM runs, torch.cuda.empty_cache(), 4-bit weights; demo peak 4.26 GB.

**Loss**: loss 1.97 → 0.97 (lowest 0.89 at step 70)".

**Limitations:** Small 3B model, short training run, only two test questions, no accuracy benchmark.